# SCBE Operator Specialty - Colab L4/T4

Trains a LoRA adapter on the agentic-operator coding lane:
- `t_operator_v1` (T-operator SFT lane)
- `eml_operator_v1` (EML-operator)
- `operator_agent_bus_extracted_v1_train` (agent bus operator extraction)
- `command_lattice_seed_train` (command lattice)
- `aligned_foundations_train` (aligned operator foundations)

Base: `Qwen/Qwen2.5-Coder-0.5B-Instruct`. Adapter target: `issdandavis/scbe-operator-specialty-qwen-colab-v1`.

Runtime: L4 (preferred) or T4. Wall budget ~8h.

Eval holdout: 10% of records held out for periodic eval (eval_steps=30) so we can see whether the adapter generalises beyond the training set.

## 1. Install dependencies

In [ ]:
!pip install -q 'transformers>=4.49' 'peft>=0.14' 'trl>=0.19' 'bitsandbytes>=0.45' 'accelerate>=1.3' 'datasets>=3.3' 'huggingface_hub>=0.27'

## 2. Authenticate with HuggingFace

Either set the `HF_TOKEN` Colab secret (recommended, padlock icon in left sidebar), or paste the token inline below.

In [ ]:
import os
from huggingface_hub import login

tok = None
try:
    from google.colab import userdata
    tok = userdata.get('HF_TOKEN')
except Exception:
    pass

if not tok:
    tok = os.environ.get('HF_TOKEN')

assert tok, 'Set HF_TOKEN as a Colab secret (padlock icon, left sidebar) or os.environ before running this cell.'
login(token=tok)
os.environ['HF_TOKEN'] = tok
print('HF authenticated')

## 3. Configuration

In [ ]:
ROUND = 'operator-specialty-v1-colab'
BASE_MODEL = 'Qwen/Qwen2.5-Coder-0.5B-Instruct'
HF_REPO = 'issdandavis/scbe-operator-specialty-qwen-colab-v1'
HF_DATASET_REPO = 'issdandavis/scbe-aethermoore-training-data'
OUTPUT_DIR = f'/content/polly-{ROUND}'

FILES = [
    't_operator_v1.sft.jsonl',
    'eml_operator_v1.sft.jsonl',
    'operator_agent_bus_extracted_v1_train.sft.jsonl',
    'command_lattice_seed_train.sft.jsonl',
    'aligned_foundations_train.sft.jsonl',
]

EPOCHS = 1
BATCH_SIZE = 2
GRAD_ACCUM = 16
MAX_LEN = 768
MAX_STEPS = 300
LEARNING_RATE = 8e-5
MAX_RECORDS = 3400
LORA_R = 16
LORA_ALPHA = 32
LORA_DROPOUT = 0.05
EVAL_SPLIT = 0.1
EVAL_STEPS = 30

import torch
assert torch.cuda.is_available(), 'No GPU - switch to L4 or T4 runtime (Runtime > Change runtime type).'
cap = torch.cuda.get_device_capability(0)
print(f'GPU: {torch.cuda.get_device_name(0)}, sm_{cap[0]}{cap[1]}, VRAM={torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB')
assert cap[0] >= 7, 'Need sm_70+ for bnb 4-bit (L4=sm_89, T4=sm_75 are both fine).'

## 4. Load operator SFT files from HF dataset

In [ ]:
import sys, random
from pathlib import Path
from huggingface_hub import hf_hub_download
from datasets import Dataset, load_dataset

records = []
for name in FILES:
    try:
        path = Path(hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=f'sft/{name}',
            repo_type='dataset',
        ))
    except Exception as exc:
        print(f'  SKIP {name}: {exc}')
        continue

    raw = path.read_text(encoding='utf-8').strip()
    if not raw or raw.startswith('version https://git-lfs'):
        print(f'  SKIP {name} (empty/LFS)')
        continue

    ds = load_dataset('json', data_files=str(path), split='train')
    cols = ds.column_names
    count = 0
    for row in ds:
        rec = None
        if 'messages' in cols and row.get('messages'):
            rec = {'messages': row['messages']}
        elif 'instruction' in cols:
            u = row.get('instruction', '')
            a = row.get('response') or row.get('output') or row.get('positive', '')
            if u and a:
                rec = {'messages': [{'role': 'user', 'content': u}, {'role': 'assistant', 'content': a}]}
        elif 'prompt' in cols:
            u = row.get('prompt', '')
            a = row.get('ideal_contains') or row.get('response', '')
            if u and a:
                rec = {'messages': [{'role': 'user', 'content': u}, {'role': 'assistant', 'content': str(a)}]}
        if rec:
            records.append(rec)
            count += 1
    print(f'  LOAD {name}: {count} records')

print(f'\nTotal: {len(records)} records')
assert records, 'No data loaded'

if len(records) > MAX_RECORDS:
    random.seed(42)
    records = random.sample(records, MAX_RECORDS)
    print(f'Sampled {MAX_RECORDS} records')

dataset = Dataset.from_list(records)

eval_dataset = None
if EVAL_SPLIT > 0 and len(dataset) >= 50:
    _split = dataset.train_test_split(test_size=EVAL_SPLIT, seed=42)
    dataset, eval_dataset = _split['train'], _split['test']
    print(f'Eval holdout: {len(eval_dataset)} records ({EVAL_SPLIT:.0%}); train: {len(dataset)}')

print(f'Dataset ready: {len(dataset)} train rows' + (f', {len(eval_dataset)} eval rows' if eval_dataset is not None else ''))

## 5. Load model with bnb 4-bit NF4 + attach LoRA

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

has_bf16 = cap[0] >= 8
compute_dtype = torch.bfloat16 if has_bf16 else torch.float16

quant_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type='nf4',
    bnb_4bit_compute_dtype=compute_dtype,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    torch_dtype=compute_dtype,
    device_map='auto',
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
model = get_peft_model(model, LoraConfig(
    r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT, bias='none',
    task_type='CAUSAL_LM',
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
))
model.print_trainable_parameters()

## 6. Train

In [ ]:
from trl import SFTConfig, SFTTrainer

_sft_kwargs = dict(
    output_dir=OUTPUT_DIR,
    hub_model_id=HF_REPO,
    push_to_hub=True,
    learning_rate=LEARNING_RATE,
    per_device_train_batch_size=BATCH_SIZE,
    gradient_accumulation_steps=GRAD_ACCUM,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    warmup_ratio=0.03,
    weight_decay=0.01,
    max_grad_norm=0.3,
    lr_scheduler_type='cosine',
    logging_steps=10,
    save_strategy='steps',
    save_steps=60,
    save_total_limit=3,
    max_length=MAX_LEN,
    packing=False,
    dataset_num_proc=1,
    report_to='none',
    fp16=not has_bf16,
    bf16=has_bf16,
    optim='adamw_torch',
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={'use_reentrant': False},
)
if eval_dataset is not None:
    _sft_kwargs.update(dict(
        eval_strategy='steps',
        eval_steps=EVAL_STEPS,
        per_device_eval_batch_size=max(1, BATCH_SIZE),
    ))

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(**_sft_kwargs),
)
trainer.train()

## 7. Save adapter and push to HuggingFace

In [ ]:
trainer.save_model()
print(f'Saved to {OUTPUT_DIR}')

try:
    trainer.push_to_hub()
    print(f'Pushed to {HF_REPO}')
except Exception as exc:
    print(f'WARN push_to_hub failed: {exc}')

import json
with open(f'{OUTPUT_DIR}/DONE.json', 'w') as f:
    json.dump({'round': ROUND, 'status': 'complete', 'hf_repo': HF_REPO}, f)
print('=== TRAINING COMPLETE ===')